# Adam Optimizer

**难度:** Medium

从零实现 Adam 优化器。

Adam 结合了动量（一阶矩）和 RMSProp（二阶矩），并通过偏差校正实现自适应的逐参数学习率。

**签名:** `MyAdam(params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8)`

**方法:**
- `step()` — 使用偏差校正后的矩更新参数
- `zero_grad()` — 将所有参数梯度清零

**约束:**
- 必须与 `torch.optim.Adam` 数值一致
- 偏差校正: `m_hat = m / (1 - beta1^t)`

**提示:**
1. m = β1·m + (1-β1)·g,  v = β2·v + (1-β2)·g²
2. 偏差校正：m̂ = m/(1-β1ᵗ),  v̂ = v/(1-β2ᵗ)
3. p -= lr · m̂ / (√v̂ + ε)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [ ]:
# ✅ SOLUTION

# ---- Step 1: 初始化 ----
class MyAdam:
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8):
        # 将生成器转为列表，方便后续按索引访问
        self.params = list(params)
        self.lr = lr          # 学习率 α
        self.beta1, self.beta2 = betas  # β1=0.9 控制动量，β2=0.999 控制 RMSProp
        self.eps = eps        # 防止除零的小常数
        self.t = 0            # 全局步数计数器（用于偏差校正）

        # 为每个参数创建对应的一阶矩 m 和二阶矩 v，初始为零
        # m 跟踪梯度的指数移动平均（类似动量）
        # v 跟踪梯度平方的指数移动平均（类似 RMSProp）
        self.m = [torch.zeros_like(p) for p in self.params]
        self.v = [torch.zeros_like(p) for p in self.params]

    # ---- Step 2: 参数更新 ----
    def step(self):
        self.t += 1  # 每次 step 递增计数器
        with torch.no_grad():  # 更新参数时不需要计算梯度图
            for i, p in enumerate(self.params):
                if p.grad is None:
                    continue  # 跳过没有梯度的参数

                g = p.grad  # 当前梯度

                # ---- Step 2a: 更新一阶矩（动量） ----
                # m_t = β1 * m_{t-1} + (1 - β1) * g_t
                # 这是梯度的指数加权移动平均，β1 越大越平滑
                self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * g

                # ---- Step 2b: 更新二阶矩（RMSProp） ----
                # v_t = β2 * v_{t-1} + (1 - β2) * g_t²
                # 这是梯度平方的指数加权移动平均，自适应调整学习率
                self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * g ** 2

                # ---- Step 2c: 偏差校正 ----
                # 为什么需要？因为 m 和 v 初始化为 0，前几步会严重偏小
                # 例如 t=1 时：m = 0.1*g（而非 g），除以 (1-0.9^1)=0.1 后恢复为 g
                # 公式：m̂ = m_t / (1 - β1^t)
                m_hat = self.m[i] / (1 - self.beta1 ** self.t)
                v_hat = self.v[i] / (1 - self.beta2 ** self.t)

                # ---- Step 2d: 参数更新 ----
                # p -= lr * m̂ / (√v̂ + ε)
                # m̂ 提供更新方向（带动量），√v̂ 自适应缩放步长
                # 梯度大的方向 v̂ 大 → 步长小（稳定），反之步长大（加速）
                p -= self.lr * m_hat / (torch.sqrt(v_hat) + self.eps)

    # ---- Step 3: 梯度清零 ----
    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()  # inplace 清零，避免梯度累积

In [ ]:
# Demo
torch.manual_seed(0)
w = torch.randn(4, 3, requires_grad=True)
opt = MyAdam([w], lr=0.01)
for i in range(5):
    loss = (w ** 2).sum()
    loss.backward()
    opt.step()
    opt.zero_grad()
    print(f'Step {i}: loss={loss.item():.4f}')

In [ ]:
from torch_judge import check
check('adam')

## 📝 核心思路总结

1. **Adam = 动量 + RMSProp + 偏差校正**：一阶矩 m 跟踪梯度均值，二阶矩 v 跟踪梯度方差
2. **偏差校正是关键**：初始 m=v=0 导致前几步估计偏小，除以 (1-β^t) 修正
3. **逐参数自适应学习率**：梯度大的参数 v 大→更新步长小，反之亦然